# Islamic Mute: train the women's-singing detector

Fine-tunes EfficientAT `mn04` (MobileNetV3, AudioSet-pretrained, MIT licence, ~1M parameters) to tell, from 1 s
of *separated voice*, whether a woman's voice is present and whether it is speech, singing or recitation.
The app (optionally, for men who enable it) mutes the voice when P(female) x P(singing) stays above the threshold
chosen here on held-out singers.  OpenSinger (CC BY-NC) is used **only** as an unseen-singer test, never for training.

Outputs in `/kaggle/working/out`: `detector_v1.onnx` (input: 769x100 power spectrogram, output: 4 probabilities),
`detector_eval.json`, `best.pt`.

In [ ]:
TRAIN_MINUTES = 90
HOURS = 2.0          # training audio per (style, gender, source)
USE_QURAN = True
import os, subprocess, time
CODE, TMP, OUT = "/kaggle/working/code", "/kaggle/tmp", "/kaggle/working/out"
for d in (CODE, TMP, OUT, CODE + "/kaggle", CODE + "/models/detector/weights"):
    os.makedirs(d, exist_ok=True)

def run(cmd, cwd=CODE, env=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
%%writefile /kaggle/working/code/kaggle/prepare_data.py
"""Prepare training audio on Kaggle: decode once to 48 kHz mono 16-bit WAV clips (fast random reads).

Sources (licences in models/voice/README.md):
  voice/speech_en   LibriSpeech (attached Kaggle dataset; test-clean/test-other excluded)      CC BY 4.0
  voice/speech_ar   Arabic Speech Corpus (Hugging Face tunis-ai/arabic_speech_corpus)           CC BY 4.0
  voice/singing     VocalSet (Hugging Face Bill13579/vocalset-mirror)                           CC BY 4.0
  voice/recitation  Diverse Quran Reciters (attached Kaggle dataset)          rights belong to the reciters
  music/slakh       Slakh2100 mixes (attached Kaggle dataset) or the 1 GB stems+mixes subset tvergho/slakh   CC BY 4.0
Clips used by our evaluation set (testset/fetch_sources.py) are skipped.

Usage: python prepare_data.py --out /kaggle/tmp/data [--hours speech_en=3,speech_ar=3,singing=4,recitation=4,slakh=10]
"""
import argparse, glob, io, os, random, subprocess, sys, tempfile
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import soundfile as sf

SR = 48000
CLIP_S = 20
EXCLUDE = {  # rows used by the evaluation set
    "speech_ar": {0, 50, 100, 150},
    "singing": {0, 200, 900, 1500, 2300, 3100},
}


def decode(src, data=None):
    """ffmpeg -> mono float32 at 48 kHz. src is a path, or a name hint when data (bytes) is given."""
    if data is not None:
        with tempfile.NamedTemporaryFile(suffix=os.path.splitext(src)[1] or ".bin") as t:
            t.write(data)
            t.flush()
            return decode(t.name)
    out = subprocess.run(["ffmpeg", "-v", "error", "-i", src, "-ac", "1", "-ar", str(SR), "-f", "f32le", "-"],
                         capture_output=True)
    return np.frombuffer(out.stdout, np.float32).copy() if out.returncode == 0 else np.zeros(0, np.float32)


def trim(x, thr_db=-50):
    if not len(x):
        return x
    e = np.abs(x)
    idx = np.where(e > np.max(e) * 10 ** (thr_db / 20))[0]
    return x[idx[0]:idx[-1] + 1] if len(idx) else x[:0]


def write_clips(x, out_dir, stem):
    """Split into <= CLIP_S second clips, normalise peak, save as 16-bit WAV. Returns seconds written."""
    x = trim(x)
    if len(x) < SR:  # skip < 1 s
        return 0.0
    os.makedirs(out_dir, exist_ok=True)
    total = 0.0
    for k, s in enumerate(range(0, len(x), CLIP_S * SR)):
        c = x[s:s + CLIP_S * SR]
        if len(c) < SR:
            continue
        c = c / (np.max(np.abs(c)) + 1e-9) * 0.9
        sf.write(os.path.join(out_dir, f"{stem}_{k}.wav"), c, SR, subtype="PCM_16")
        total += len(c) / SR
    return total


def job_file(args):
    path, out_dir, stem = args
    return write_clips(decode(path), out_dir, stem)


INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")


def find(pattern, hint):
    """Search only inside attached datasets whose folder name contains `hint`
    (works for both /kaggle/input/<slug> and /kaggle/input/datasets/<owner>/<slug> layouts)."""
    roots = [d for d in glob.glob(f"{INPUT}/*") + glob.glob(f"{INPUT}/*/*/*") if hint in os.path.basename(d).lower()]
    out = set()
    for r in roots:
        out.update(glob.glob(os.path.join(r, "**", pattern), recursive=True))
    print(f"found {len(out)} {pattern} files for '{hint}' in {roots}", flush=True)
    return sorted(out)


def from_files(files, out_dir, budget_s, workers, rng, prefix):
    rng.shuffle(files)
    done, i = 0.0, 0
    with ProcessPoolExecutor(workers) as ex:
        while done < budget_s and i < len(files):
            batch = files[i:i + workers * 4]
            i += len(batch)
            jobs = [(f, out_dir, f"{prefix}{i + k}") for k, f in enumerate(batch)]
            done += sum(ex.map(job_file, jobs))
    return done


def from_hf_parquet(repo, out_dir, budget_s, exclude, rng, split_hint="train", columns=("audio",)):
    """Stream rows from a Hugging Face parquet dataset (audio columns with bytes)."""
    import pyarrow.parquet as pq
    from huggingface_hub import HfApi, hf_hub_download
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.endswith(".parquet") and split_hint in f)
    done, row = 0.0, 0
    for f in files:
        local = hf_hub_download(repo, f, repo_type="dataset")
        table = pq.ParquetFile(local)
        for batch in table.iter_batches(batch_size=32, columns=list(columns)):
            cols = [batch.column(i).to_pylist() for i in range(len(columns))]
            for vals in zip(*cols):
                r, row = row, row + 1
                if r in exclude:
                    continue
                for ci, a in enumerate(vals):
                    if not a or not a.get("bytes"):
                        continue
                    done += write_clips(decode(a.get("path") or "x.wav", a["bytes"]), out_dir, f"r{r}c{ci}")
                if done >= budget_s:
                    os.remove(local)
                    return done
        os.remove(local)  # keep disk use low
    return done


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="/kaggle/tmp/data")
    ap.add_argument("--hours", default="speech_en=3,speech_ar=3,singing=4,recitation=4,slakh=10")
    ap.add_argument("--no-quran", action="store_true", help="skip recitation data (rights not cleared)")
    ap.add_argument("--workers", type=int, default=os.cpu_count() or 4)
    args = ap.parse_args()
    hours = {k: float(v) for k, v in (kv.split("=") for kv in args.hours.split(","))}
    rng = random.Random(0)
    report = {}

    libri = [f for f in find("*.flac", "librispeech") if "/test-" not in f]
    report["speech_en"] = from_files(libri, f"{args.out}/voice/speech_en", hours["speech_en"] * 3600, args.workers, rng, "ls")

    report["speech_ar"] = from_hf_parquet("tunis-ai/arabic_speech_corpus", f"{args.out}/voice/speech_ar",
                                          hours["speech_ar"] * 3600, EXCLUDE["speech_ar"], rng)
    report["singing"] = from_hf_parquet("Bill13579/vocalset-mirror", f"{args.out}/voice/singing",
                                        hours["singing"] * 3600, EXCLUDE["singing"], rng)
    if not args.no_quran:
        quran = find("*.mp3", "quran")
        report["recitation"] = from_files(quran, f"{args.out}/voice/recitation", hours["recitation"] * 3600,
                                          args.workers, rng, "q")

    # music: full Slakh mixes plus random partial sub-mixes (fewer instruments = harder, more varied)
    mixes = [f for f in find("mix.flac", "slakh") if "/omitted/" not in f and "/test/" not in f]
    if mixes:  # full Slakh2100 attached as a Kaggle dataset
        report["slakh"] = from_files(mixes, f"{args.out}/music/slakh", hours["slakh"] * 3600, args.workers, rng, "s")
    else:  # 1 GB Slakh subset on Hugging Face: single instrument stems plus their mixes
        report["slakh"] = from_hf_parquet("tvergho/slakh", f"{args.out}/music/slakh", hours["slakh"] * 3600, set(),
                                          rng, split_hint="test", columns=("audio", "stem_mix"))

    for k, v in report.items():
        print(f"{k:12s} {v / 3600:6.2f} h")
    missing = [k for k, v in report.items() if v < 600]
    if missing:
        sys.exit(f"too little data for: {missing} (check the attached datasets)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/models/detector/mobilenet.py
"""Our own implementation of the EfficientAT MobileNetV3 audio tagger (architecture from Schmid et al.,
"Efficient Large-Scale Audio Tagging via Transformer-to-CNN Knowledge Distillation", ICASSP 2023;
pretrained AudioSet weights are MIT-licensed, see efficientat/LICENSE). Parameter names match the
released checkpoints so they load with torch.load(weights_only=True) - no pickled code is executed.

Front end: the reference runs at 32 kHz (1024-pt STFT, 800-sample Hann window, hop 320). We run it at the
engine's 48 kHz with the same time/frequency grid (1536-pt STFT, 1200-sample window, hop 480 = 10 ms, so
31.25 Hz bins either way), so each 10 ms engine hop yields exactly one mel frame and nothing is resampled.
128 Kaldi mel bands 0-15 kHz, pre-emphasis 0.97, log(x+1e-5), (x+4.5)/5.  Frames are NOT centre-padded:
frame k covers samples [k*HOP, k*HOP+N_FFT), exactly what the C streaming front end computes.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchaudio.compliance.kaldi import get_mel_banks

SR, N_FFT, WIN, HOP, N_MELS = 48000, 1536, 1200, 480, 128
FRAMES = 100                         # 1 s analysis window
WINDOW_SAMPLES = (FRAMES - 1) * HOP + N_FFT + 1   # +1 for the pre-emphasis tap


def make_divisible(v, d=8):
    n = max(d, int(v + d / 2) // d * d)
    return n + d if n < 0.9 * v else n


class CNA(nn.Sequential):
    """conv (0) + batch norm (1) + optional activation (2)."""

    def __init__(self, cin, cout, k=1, stride=1, groups=1, act=None):
        layers = [nn.Conv2d(cin, cout, k, stride, (k - 1) // 2, groups=groups, bias=False),
                  nn.BatchNorm2d(cout, eps=0.001, momentum=0.01)]
        if act is not None:
            layers.append(act())
        super().__init__(*layers)


class SE(nn.Module):
    """Channel squeeze-excitation (EfficientAT 'ConcurrentSEBlock' with se_dims='c')."""

    def __init__(self, c, r=4):
        super().__init__()
        s = make_divisible(c // r)
        self.conc_se_layers = nn.ModuleList([nn.Module()])
        self.conc_se_layers[0].fc1 = nn.Linear(c, s)
        self.conc_se_layers[0].fc2 = nn.Linear(s, c)

    def forward(self, x):
        se = self.conc_se_layers[0]
        w = torch.sigmoid(se.fc2(F.relu(se.fc1(x.mean((2, 3))))))
        return x * w[:, :, None, None]


class Block(nn.Module):
    def __init__(self, cin, k, cexp, cout, use_se, hs, stride):
        super().__init__()
        act = nn.Hardswish if hs else nn.ReLU
        layers = []
        if cexp != cin:
            layers.append(CNA(cin, cexp, 1, act=act))
        layers.append(CNA(cexp, cexp, k, stride, groups=cexp, act=act))
        if use_se:
            layers.append(SE(cexp))
        layers.append(CNA(cexp, cout, 1))
        self.block = nn.Sequential(*layers)
        self.res = stride == 1 and cin == cout

    def forward(self, x):
        y = self.block(x)
        return y + x if self.res else y


# input, kernel, expanded, out, SE, hard-swish, stride  (MobileNetV3-Large)
CONF = [(16, 3, 16, 16, 0, 0, 1), (16, 3, 64, 24, 0, 0, 2), (24, 3, 72, 24, 0, 0, 1), (24, 5, 72, 40, 1, 0, 2),
        (40, 5, 120, 40, 1, 0, 1), (40, 5, 120, 40, 1, 0, 1), (40, 3, 240, 80, 0, 1, 2), (80, 3, 200, 80, 0, 1, 1),
        (80, 3, 184, 80, 0, 1, 1), (80, 3, 184, 80, 0, 1, 1), (80, 3, 480, 112, 1, 1, 1),
        (112, 3, 672, 112, 1, 1, 1), (112, 5, 672, 160, 1, 1, 2), (160, 5, 960, 160, 1, 1, 1),
        (160, 5, 960, 160, 1, 1, 1)]


class MN(nn.Module):
    def __init__(self, width=1.0, num_classes=527):
        super().__init__()
        a = lambda c: make_divisible(c * width)  # noqa: E731
        layers = [CNA(1, a(16), 3, 2, act=nn.Hardswish)]
        for cin, k, ce, co, se, hs, s in CONF:
            layers.append(Block(a(cin), k, a(ce), a(co), se, hs, s))
        last_in = a(160)
        layers.append(CNA(last_in, 6 * last_in, 1, act=nn.Hardswish))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1), nn.Linear(6 * last_in, a(1280)),
                                        nn.Hardswish(), nn.Dropout(0.2), nn.Linear(a(1280), num_classes))

    def embed(self, mel):  # mel: (B, 128, T) -> (B, C) pooled features
        return self.features(mel.unsqueeze(1)).mean((2, 3))

    def forward(self, mel):
        return self.classifier(self.features(mel.unsqueeze(1)))


class MelFrontEnd(nn.Module):
    def __init__(self, sr=SR, n_fft=N_FFT, win=WIN, hop=HOP, center=False):
        super().__init__()
        self.n_fft, self.hop, self.center = n_fft, hop, center
        fb, _ = get_mel_banks(N_MELS, n_fft, float(sr), 0.0, 15000.0, 100.0, -500.0, 1.0)
        self.register_buffer("fb", F.pad(fb, (0, 1)), persistent=False)          # (128, 513)
        w = torch.hann_window(win, periodic=False)
        self.register_buffer("window", F.pad(w, ((n_fft - win) // 2,) * 2), persistent=False)

    def forward(self, x):  # x: (B, n) -> (B, 128, frames)
        x = x[:, 1:] - 0.97 * x[:, :-1]
        X = torch.stft(x, self.n_fft, self.hop, self.n_fft, self.window, center=self.center, return_complex=True)
        return (torch.log(self.fb @ X.abs().pow(2) + 1e-5) + 4.5) / 5.0


def load_pretrained(path, width):
    m = MN(width)
    sd = torch.load(path, map_location="cpu", weights_only=True)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "num_batches" not in k] + [k for k in unexpected if "num_batches" not in k]
    if bad:
        raise RuntimeError(f"checkpoint mismatch: {bad[:10]}")
    return m.eval()

In [ ]:
%%writefile /kaggle/working/code/models/detector/prepare_detector_data.py
"""Build labelled audio for the women's-singing detector (runs on Kaggle; 48 kHz mono 16-bit WAV clips <= 20 s).

Layout: <out>/<split>/<style>_<gender>/*.wav   split in {train, val, ext}; style in {speech, singing, recitation}
        <out>/music/*.wav                       instrument residue used as augmentation

| source                                   | gives                          | licence           | split            |
|------------------------------------------|--------------------------------|-------------------|------------------|
| LibriSpeech (attached) + SPEAKERS.TXT    | English speech, F/M            | CC BY 4.0         | train / val=test-clean |
| FLEURS ar_eg (HF google/fleurs)          | Arabic speech, F/M             | CC BY 4.0         | train / val=dev  |
| VocalSet 1.1 (Zenodo 1442513)            | singing F/M (+ 'spoken' speech)| CC BY 4.0         | 2 singers held out for val |
| Diverse Quran Reciters (attached)        | recitation, M + Ayu Rumiyati F | reciters' rights  | 1 male reciter + 20% Ayu held out |
| Slakh2100 subset (HF tvergho/slakh)      | instrument residue             | CC BY 4.0         | augmentation     |
| OpenSinger (HF CodecSR/opensinger_*)     | Chinese singing F/M            | CC BY-NC-SA       | ext (evaluation only, never trained on) |
"""
import argparse, csv, glob, io, os, random, re, sys, tarfile, urllib.request, zipfile

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(HERE)), "kaggle"))
from prepare_data import decode, write_clips, find, from_files, from_hf_parquet  # noqa: E402

VOCALSET_URL = "https://zenodo.org/api/records/1442513/files/VocalSet11.zip/content"


def budget(files, out_dir, hours, workers, rng, prefix):
    if not files:
        print("  no files for", out_dir, flush=True)
        return 0.0
    s = from_files(list(files), out_dir, hours * 3600, workers, rng, prefix)
    print(f"  {out_dir}: {s / 3600:.2f} h", flush=True)
    return s


def librispeech(out, h, workers, rng):
    spk = {}
    for p in find("SPEAKERS.TXT", "librispeech"):
        for line in open(p, errors="ignore"):
            if line.startswith(";"):
                continue
            f = [c.strip() for c in line.split("|")]
            if len(f) >= 2 and f[0].isdigit() and f[1] in ("F", "M"):
                spk[f[0]] = f[1]
    files = find("*.flac", "librispeech")
    print(f"LibriSpeech: {len(spk)} speakers with sex labels, {len(files)} files", flush=True)
    for split, keep in (("train", lambda f: "/test-" not in f), ("val", lambda f: "/test-clean/" in f)):
        for g in "FM":
            sel = [f for f in files if keep(f) and spk.get(os.path.basename(f).split("-")[0]) == g]
            budget(sel, f"{out}/{split}/speech_{g}", h if split == "train" else 0.3, workers, rng, f"ls{g}")


def fleurs(out, lang, h, tmp):
    from huggingface_hub import hf_hub_download
    for split, hours, dst in (("train", h, "train"), ("dev", 0.3, "val")):
        tsv = hf_hub_download("google/fleurs", f"data/{lang}/{split}.tsv", repo_type="dataset", cache_dir=tmp)
        gender = {}
        for r in csv.reader(open(tsv, encoding="utf-8"), delimiter="\t", quoting=csv.QUOTE_NONE):
            if len(r) >= 7 and r[-1] in ("FEMALE", "MALE"):
                gender[r[1]] = r[-1][0]
        tar = hf_hub_download("google/fleurs", f"data/{lang}/audio/{split}.tar.gz", repo_type="dataset", cache_dir=tmp)
        done = {"F": 0.0, "M": 0.0}
        with tarfile.open(tar) as t:
            for m in t:
                g = gender.get(os.path.basename(m.name))
                if not m.isfile() or g is None or done[g] >= hours * 3600:
                    continue
                x = decode(m.name, t.extractfile(m).read())
                done[g] += write_clips(x, f"{out}/{dst}/speech_{g}", f"fl{lang}_{os.path.basename(m.name)[:-4]}")
        os.remove(tar)
        print(f"FLEURS {lang} {split}: F {done['F'] / 3600:.2f} h, M {done['M'] / 3600:.2f} h", flush=True)


def singer(path):
    """VocalSet singer id as 'f3' / 'm11'. Folders are 'female3' / 'male11' (FULL/female3/...);
    file names start with 'f3_' / 'm11_'. Returns None if neither is found."""
    for p in path.split("/")[:-1]:
        m = re.fullmatch(r"(female|male|[fm])(\d+)", p.lower())
        if m:
            return m.group(1)[0] + m.group(2)
    m = re.match(r"([fm])(\d+)_", os.path.basename(path).lower())
    return m.group(1) + m.group(2) if m else None


def vocalset(out, h, tmp, rng):
    z = f"{tmp}/VocalSet11.zip"
    if not os.path.exists(z):
        print("downloading VocalSet 1.1 (2.1 GB) from Zenodo", flush=True)
        urllib.request.urlretrieve(VOCALSET_URL, z)
    with zipfile.ZipFile(z) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".wav") and "__macosx" not in n.lower()]
        singers = sorted({singer(n) for n in names if singer(n)}, key=lambda s: (s[0], int(s[1:])))
        if not all(any(s[0] == g for s in singers) for g in "fm"):
            raise SystemExit(f"VocalSet: could not read singer ids (got {singers}); sample paths: {names[:5]}")
        held = {max((s for s in singers if s[0] == g), key=lambda s: int(s[1:])) for g in "fm"}
        print(f"VocalSet: {len(names)} wavs, singers {singers}, validation singers {sorted(held)}", flush=True)
        rng.shuffle(names)
        done = {}
        for n in names:
            s = singer(n)
            if not s:
                continue
            style = "speech" if "spoken" in n.lower() else "singing"
            split = "val" if s in held else "train"
            key = (split, style, s[0].upper())
            cap = (h if split == "train" else 0.4) * 3600 * (0.15 if style == "speech" else 1.0)
            if done.get(key, 0.0) >= cap:
                continue
            x = decode(n, zf.read(n))
            done[key] = done.get(key, 0.0) + write_clips(x, f"{out}/{split}/{style}_{s[0].upper()}",
                                                         f"vs_{os.path.basename(n)[:-4]}")
    os.remove(z)
    for k, v in sorted(done.items()):
        print(f"  VocalSet {k}: {v / 3600:.2f} h", flush=True)


def quran(out, h, workers, rng):
    files = find("*.mp3", "quran")
    fem = [f for f in files if "ayu_rumiyati" in f.lower()]
    male = [f for f in files if f not in set(fem)]
    reciters = sorted({os.path.basename(os.path.dirname(f)) for f in male})
    held = reciters[len(reciters) // 2] if reciters else None
    print(f"Quran: {len(fem)} female files, {len(male)} male files, {len(reciters)} male reciters; held out {held}", flush=True)
    rng.shuffle(fem)
    k = max(1, len(fem) // 5)
    budget(fem[k:], f"{out}/train/recitation_F", h, workers, rng, "qF")
    budget(fem[:k], f"{out}/val/recitation_F", 0.3, workers, rng, "qFv")
    budget([f for f in male if os.path.basename(os.path.dirname(f)) != held], f"{out}/train/recitation_M", h, workers, rng, "qM")
    budget([f for f in male if os.path.basename(os.path.dirname(f)) == held], f"{out}/val/recitation_M", 0.3, workers, rng, "qMv")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="/kaggle/tmp/det")
    ap.add_argument("--tmp", default="/kaggle/tmp/dl")
    ap.add_argument("--hours", type=float, default=2.0, help="training hours per (style, gender) and source")
    ap.add_argument("--no-quran", action="store_true")
    ap.add_argument("--workers", type=int, default=os.cpu_count() or 4)
    a = ap.parse_args()
    os.makedirs(a.tmp, exist_ok=True)
    rng = random.Random(0)
    librispeech(a.out, a.hours, a.workers, rng)
    fleurs(a.out, "ar_eg", a.hours, a.tmp)
    vocalset(a.out, a.hours * 2, a.tmp, rng)
    if not a.no_quran:
        quran(a.out, a.hours, a.workers, rng)
    from_hf_parquet("tvergho/slakh", f"{a.out}/music", 1.0 * 3600, set(), rng, columns=("audio", "stem_mix"))
    for g, repo in (("F", "CodecSR/opensinger_female"), ("M", "CodecSR/opensinger_male")):
        s = from_hf_parquet(repo, f"{a.out}/ext/singing_{g}", 0.4 * 3600, set(), rng, split_hint="original")
        print(f"OpenSinger {g} (evaluation only): {s / 3600:.2f} h", flush=True)
    for d in sorted(glob.glob(f"{a.out}/*/*")) + [f"{a.out}/music"]:
        print(f"{d}: {len(glob.glob(d + '/*.wav'))} clips", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/models/detector/train_detector.py
"""Fine-tune the women's-singing detector: EfficientAT mn04 (AudioSet-pretrained, MIT) + a small head.

Outputs per 1 s window of the *separated voice*: P(a female voice is present), and P(style) over
{speech, singing, recitation}.  The app mutes when P(female) * P(singing) stays above a threshold
(chosen here on held-out singers for a target catch rate).

python train_detector.py --data /kaggle/tmp/det --init weights/mn04_as_mAP_432.pt --out out --minutes 60
"""
import argparse, glob, json, math, os, random, sys, time

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from mobilenet import MN, MelFrontEnd, WINDOW_SAMPLES, SR, N_FFT, WIN, HOP, N_MELS, FRAMES  # noqa: E402

STYLES = ["speech", "singing", "recitation"]
OUT_NAMES = ["female", "speech", "singing", "recitation"]


# ----------------------------------------------------------------------------- data
class Pool:
    def __init__(self, files):
        self.files = files
        self.frames = [sf.info(f).frames for f in files]

    def __len__(self):
        return len(self.files)

    def crop(self, rng, n):
        for _ in range(10):
            i = rng.randrange(len(self.files))
            if self.frames[i] >= n:
                o = rng.randrange(self.frames[i] - n + 1)
                x, _ = sf.read(self.files[i], start=o, frames=n, dtype="float32", always_2d=True)
                x = x.mean(1)
                if np.sqrt(np.mean(x ** 2)) > 1e-3:
                    return x
        x, _ = sf.read(self.files[rng.randrange(len(self.files))], dtype="float32", always_2d=True)
        x = np.resize(x.mean(1), n)
        return x


def pools(root):
    out = {}
    for d in glob.glob(os.path.join(root, "*_*")):
        style, g = os.path.basename(d).split("_")
        fs = sorted(glob.glob(d + "/*.wav"))
        if fs and style in STYLES:
            out[(style, g)] = Pool(fs)
    return out


def rms(x):
    return np.sqrt(np.mean(x ** 2) + 1e-12)


def lowpass(x, rng):
    X = np.fft.rfft(x)
    fc = rng.uniform(4000, 14000)
    f = np.fft.rfftfreq(len(x), 1 / SR)
    X *= 1 / (1 + (f / fc) ** 8)
    return np.fft.irfft(X, len(x)).astype(np.float32)


def reverb(x, rng):
    t60 = rng.uniform(0.2, 1.2)
    n = int(t60 * SR)
    ir = rng_noise(rng, n) * np.exp(-6.9 * np.arange(n) / n)
    ir[0] = 1.0
    y = np.fft.irfft(np.fft.rfft(x, len(x) + n) * np.fft.rfft(ir, len(x) + n))[: len(x)]
    return (y / (rms(y) + 1e-9) * rms(x)).astype(np.float32)


def rng_noise(rng, n):
    return np.random.default_rng(rng.randrange(1 << 30)).standard_normal(n).astype(np.float32)


class Batches(torch.utils.data.IterableDataset):
    """Random 1 s examples with labels (female, style)."""

    def __init__(self, root, music_dir, seed, mix_p=0.25):
        self.root, self.music_dir, self.seed, self.mix_p = root, music_dir, seed, mix_p

    def __iter__(self):
        wi = torch.utils.data.get_worker_info()
        rng = random.Random(self.seed + (wi.id if wi else 0) * 7919)
        P = pools(self.root)
        music = Pool(sorted(glob.glob(self.music_dir + "/*.wav"))) if self.music_dir else None
        n = WINDOW_SAMPLES
        style_p = {"speech": 0.35, "singing": 0.45, "recitation": 0.20}
        styles = [s for s in STYLES if any(k[0] == s for k in P)]
        while True:
            style = rng.choices(styles, [style_p[s] for s in styles])[0]
            gs = [g for g in "FM" if (style, g) in P]
            g = rng.choice(gs)
            x = P[(style, g)].crop(rng, n)
            x = x / (rms(x) + 1e-9) * 0.1
            female = g == "F"
            if rng.random() < self.mix_p:  # a second voice: duets, backing vocals, overlapping speech
                g2 = rng.choice([h for h in "FM" if (style, h) in P])
                y = P[(style, g2)].crop(rng, n)
                y = y / (rms(y) + 1e-9) * 0.1 * 10 ** (rng.uniform(-12, 0) / 20)
                x = x + y
                female = female or g2 == "F"
            if rng.random() < 0.3:
                x = lowpass(x, rng)
            if rng.random() < 0.25:
                x = reverb(x, rng)
            if music is not None and rng.random() < 0.7:  # imperfect separation leaves some instruments
                m = music.crop(rng, n)
                x = x + m / (rms(m) + 1e-9) * rms(x) * 10 ** (-rng.uniform(10, 35) / 20)
            if rng.random() < 0.1:
                x = x + rng_noise(rng, n) * rms(x) * 10 ** (-rng.uniform(25, 50) / 20)
            x = x * 10 ** (rng.uniform(-35, 0) / 20) / (np.max(np.abs(x)) + 1e-9)
            if rng.random() < 0.2:  # voice onset after silence: the streaming window is partly empty
                x[: int(rng.uniform(0, 0.75) * n)] = 0.0
            yield torch.from_numpy(x.astype(np.float32)), torch.tensor(float(female)), torch.tensor(STYLES.index(style))


def fixed_set(root, per_class, seed=1):
    """Deterministic validation windows: (audio, female, style, group)."""
    rng = random.Random(seed)
    xs, fe, st, grp = [], [], [], []
    for (style, g), pool in sorted(pools(root).items()):
        for _ in range(per_class):
            x = pool.crop(rng, WINDOW_SAMPLES)
            xs.append(x / (np.max(np.abs(x)) + 1e-9) * 0.5)
            fe.append(float(g == "F"))
            st.append(STYLES.index(style))
            grp.append(f"{style}_{g}")
    return torch.from_numpy(np.stack(xs)), torch.tensor(fe), torch.tensor(st), grp


# ----------------------------------------------------------------------------- model
class Detector(nn.Module):
    """power spectrogram (B, 769, 100) -> (B, 4) probabilities [female, speech, singing, recitation]."""

    def __init__(self, width=0.4):
        super().__init__()
        self.front = MelFrontEnd()
        self.net = MN(width)
        c = self.net.classifier[2].in_features
        self.head = nn.Sequential(nn.Linear(c, 128), nn.Hardswish(), nn.Dropout(0.2), nn.Linear(128, 4))

    def logits_from_power(self, power):
        mel = (torch.log(self.front.fb @ power + 1e-5) + 4.5) / 5.0
        e = self.net.features(mel.unsqueeze(1)).mean((2, 3))
        return self.head(e)

    def power(self, audio):
        x = audio[:, 1:] - 0.97 * audio[:, :-1]
        X = torch.stft(x, N_FFT, HOP, N_FFT, self.front.window, center=False, return_complex=True)
        return X.real ** 2 + X.imag ** 2

    def forward(self, audio):
        return self.logits_from_power(self.power(audio))


class Exported(nn.Module):
    def __init__(self, det):
        super().__init__()
        self.det = det

    def forward(self, power):
        z = self.det.logits_from_power(power)
        return torch.cat([torch.sigmoid(z[:, :1]), torch.softmax(z[:, 1:], 1)], 1)


def auc(pos, neg):
    pos, neg = np.asarray(pos), np.asarray(neg)
    if not len(pos) or not len(neg):
        return float("nan")
    r = np.argsort(np.argsort(np.concatenate([pos, neg]))) + 1
    return float((r[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


@torch.no_grad()
def evaluate(model, vs, dev, target_catch=0.95):
    model.eval()
    x, fe, st, grp = vs
    probs = torch.cat([Exported(model)(model.power(x[i:i + 128].to(dev))).cpu() for i in range(0, len(x), 128)])
    model.train()
    score = (probs[:, 0] * probs[:, 2]).numpy()  # P(female) * P(singing)
    grp = np.array(grp)
    fs = score[grp == "singing_F"]
    neg = score[grp != "singing_F"]
    res = {"auc_female_singing_vs_rest": auc(fs, neg),
           "auc_female_vs_male_singing": auc(fs, score[grp == "singing_M"]),
           "gender_acc": float(((probs[:, 0] > 0.5).float() == fe).float().mean()),
           "style_acc": float((probs[:, 1:].argmax(1) == st).float().mean())}
    if len(fs):
        thr = float(np.quantile(fs, 1 - target_catch))
        res["threshold"] = thr
        res["false_mute_rate_by_group"] = {g: float(np.mean(score[grp == g] >= thr)) for g in sorted(set(grp)) if g != "singing_F"}
        res["catch_at_threshold"] = float(np.mean(fs >= thr))
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--init", default=os.path.join(HERE, "weights", "mn04_as_mAP_432.pt"))
    ap.add_argument("--out", required=True)
    ap.add_argument("--minutes", type=float, default=60)
    ap.add_argument("--steps", type=int, default=100000)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--val-per-class", type=int, default=200)
    ap.add_argument("--eval-every", type=int, default=1000)
    ap.add_argument("--target-catch", type=float, default=0.95)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    model = Detector()
    sd = torch.load(a.init, map_location="cpu", weights_only=True)
    missing, _ = model.net.load_state_dict(sd, strict=False)
    assert not [k for k in missing if "num_batches" not in k], missing
    model.to(dev).train()
    vs = fixed_set(os.path.join(a.data, "val"), a.val_per_class)
    ext = fixed_set(os.path.join(a.data, "ext"), a.val_per_class) if glob.glob(os.path.join(a.data, "ext", "*_*")) else None
    music = os.path.join(a.data, "music")
    dl = torch.utils.data.DataLoader(Batches(os.path.join(a.data, "train"), music if os.path.isdir(music) else None, 0),
                                     batch_size=a.batch, num_workers=a.workers, persistent_workers=a.workers > 0)
    opt = torch.optim.AdamW([{"params": model.net.parameters(), "lr": a.lr * 0.3},
                             {"params": model.head.parameters(), "lr": a.lr}], weight_decay=1e-4)
    base = [g["lr"] for g in opt.param_groups]
    t0, best, log = time.time(), -1.0, []
    for step, (x, fe, st) in enumerate(dl):
        frac = min(1.0, max(step / a.steps, (time.time() - t0) / (a.minutes * 60)))
        if frac >= 1.0:
            break
        warm = min(1.0, step / 300)
        for g, b in zip(opt.param_groups, base):
            g["lr"] = b * warm * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * frac)))
        z = model(x.to(dev))
        loss = F.binary_cross_entropy_with_logits(z[:, 0], fe.to(dev)) + F.cross_entropy(z[:, 1:], st.to(dev))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        if step % 100 == 0:
            print(f"step {step} loss {loss.item():.4f} {(time.time() - t0) / 60:.1f} min", flush=True)
        if step and step % a.eval_every == 0:
            r = evaluate(model, vs, dev, a.target_catch)
            r["step"] = step
            log.append(r)
            print(json.dumps(r), flush=True)
            if r["auc_female_singing_vs_rest"] > best:
                best = r["auc_female_singing_vs_rest"]
                torch.save(model.state_dict(), os.path.join(a.out, "best.pt"))
    torch.save(model.state_dict(), os.path.join(a.out, "last.pt"))
    if os.path.exists(os.path.join(a.out, "best.pt")):
        model.load_state_dict(torch.load(os.path.join(a.out, "best.pt"), map_location=dev, weights_only=True))
    final = {"val": evaluate(model, vs, dev, a.target_catch)}
    if ext:
        final["ext_opensinger"] = evaluate(model, ext, dev, a.target_catch)
        thr = final["val"].get("threshold")
        if thr is not None:  # how the val-chosen threshold transfers to unseen singers/language
            x, fe, st, grp = ext
            model.eval()
            with torch.no_grad():
                p = torch.cat([Exported(model)(model.power(x[i:i + 128].to(dev))).cpu() for i in range(0, len(x), 128)])
            s = (p[:, 0] * p[:, 2]).numpy()
            grp = np.array(grp)
            final["ext_opensinger"]["at_val_threshold"] = {
                "catch_female_singing": float(np.mean(s[grp == "singing_F"] >= thr)),
                "false_mute_male_singing": float(np.mean(s[grp == "singing_M"] >= thr))}
    final["train_log"] = log
    json.dump(final, open(os.path.join(a.out, "detector_eval.json"), "w"), indent=1)
    print(json.dumps({k: v for k, v in final.items() if k != "train_log"}, indent=1))
    export(model.cpu(), os.path.join(a.out, "detector_v1.onnx"), final["val"].get("threshold", 0.5))


def export(model, path, threshold):
    import onnx
    model.eval()
    ex = Exported(model).eval()
    power = torch.rand(1, N_FFT // 2 + 1, FRAMES)
    torch.onnx.export(ex, (power,), path, input_names=["power"], output_names=["probs"], opset_version=17,
                      dynamic_axes=None)
    m = onnx.load(path)
    meta = {"im.detector": "1", "im.det_sample_rate": str(SR), "im.det_n_fft": str(N_FFT), "im.det_win": str(WIN),
            "im.det_hop": str(HOP), "im.det_frames": str(FRAMES), "im.det_mels": str(N_MELS), "im.det_preemph": "0.97",
            "im.det_window": "hann_symmetric_centered", "im.det_outputs": ",".join(OUT_NAMES),
            "im.det_threshold": f"{threshold:.4f}", "im.det_eval_every_hops": "25", "im.det_release_s": "2.0",
            "im.model_name": "detector_v1 (EfficientAT mn04 fine-tuned)",
            "im.licence": "MIT (EfficientAT weights); trained on CC BY 4.0 data (LibriSpeech, FLEURS, VocalSet) + Quran recitations"}
    for k, v in meta.items():
        e = m.metadata_props.add()
        e.key, e.value = k, v
    onnx.save(m, path)
    print("exported", path, os.path.getsize(path) // 1024, "KB")
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        p = torch.rand(4, 1, N_FFT // 2 + 1, FRAMES) * 10 ** torch.empty(4, 1, 1, 1).uniform_(-6, 1)
        err = max(float(np.max(np.abs(sess.run(None, {"power": q.numpy()})[0] - ex(q).detach().numpy()))) for q in p)
        print(f"ONNX Runtime vs PyTorch max abs diff {err:.2e}")
        assert err < 1e-4, err
    except ImportError:
        pass


if __name__ == "__main__":
    main()

In [ ]:
PY = f"{TMP}/venv/bin/python"
if not os.path.exists(PY):
    run("pip install -q uv")
    run(f"uv venv -q -p 3.10 {TMP}/venv")
    run(f"uv pip install -q -p {PY} torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121")
    run(f"uv pip install -q -p {PY} numpy==1.26.4 soundfile onnx==1.16.2 onnxruntime==1.19.2 'protobuf<5' huggingface_hub pyarrow")
run("command -v ffmpeg || (apt-get update -qq && apt-get install -y -qq ffmpeg)")
run("curl -sfL -o models/detector/weights/mn04_as_mAP_432.pt https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/mn04_as_mAP_432.pt")
run("sha256sum models/detector/weights/mn04_as_mAP_432.pt")

In [ ]:
run(f"{PY} models/detector/prepare_detector_data.py --out {TMP}/det --tmp {TMP}/dl --hours {HOURS}" + ("" if USE_QURAN else " --no-quran"))
run(f"du -sh {TMP}/det/*")

In [ ]:
run(f"{PY} models/detector/train_detector.py --data {TMP}/det --out {OUT} --minutes {TRAIN_MINUTES} --workers 4")
run(f"rm -f {OUT}/last.pt; ls -la {OUT}")